In [ ]:
"""F937: runtime-safe PyTorch MLP residual anchored by F931-style stable signals."""

import time
import numpy as np
import pandas as pd
import dai

try:
    import torch
    import torch.nn as nn
except Exception:
    torch = None
    nn = None


FACTOR_ID = "F937"
FACTOR_NAME = "fast_mlp_anchor_residual"


LOOKBACK_DAYS = 720
MAX_TRAIN_ROWS = 260000
MIN_TRAIN_ROWS = 15000
BATCH_SIZE = 4096
EPOCHS = 4
TRAIN_TIME_BUDGET_SECONDS = 5400
RANDOM_STATE = 20260708


def _rank_by_date(frame, column, out_col):
    frame[out_col] = frame.groupby("date")[column].rank(pct=True, method="average") - 0.5
    return frame


def _select_feature_columns(frame):
    blocked = {"date", "instrument"}
    blocked_keywords = ("future", "label", "target", "next", "forward")
    numeric_cols = [
        col for col in frame.columns
        if col not in blocked and pd.api.types.is_numeric_dtype(frame[col])
    ]
    return [col for col in numeric_cols if not any(k in col.lower() for k in blocked_keywords)]


def _make_rank_features(frame, feature_cols):
    ranked = frame[["date", "instrument"] + feature_cols].copy()
    for col in feature_cols:
        ranked[col] = ranked.groupby("date")[col].rank(pct=True, method="average") - 0.5
    ranked[feature_cols] = ranked[feature_cols].replace([np.inf, -np.inf], np.nan)
    return ranked


if torch is not None:
    class ResidualMLP(nn.Module):
        def __init__(self, n_features, hidden_size=96, dropout=0.18):
            super().__init__()
            self.net = nn.Sequential(
                nn.LayerNorm(n_features),
                nn.Linear(n_features, hidden_size),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size, hidden_size // 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size // 2, 1),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)


def _fit_predict_mlp(train, pred, feature_cols, deadline):
    if torch is None or len(train) < MIN_TRAIN_ROWS or pred.empty:
        out = pred[["date", "instrument"]].copy()
        out["mlp_raw"] = 0.0
        return out

    if len(train) > MAX_TRAIN_ROWS:
        train = train.sample(n=MAX_TRAIN_ROWS, random_state=RANDOM_STATE)

    fill_values = train[feature_cols].median(axis=0).fillna(0.0)
    x_train = train[feature_cols].fillna(fill_values).to_numpy(dtype="float32")
    y_train = train["target"].fillna(0.0).to_numpy(dtype="float32")
    x_pred = pred[feature_cols].fillna(fill_values).to_numpy(dtype="float32")

    torch.manual_seed(RANDOM_STATE)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ResidualMLP(x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0012, weight_decay=0.02)
    loss_fn = nn.SmoothL1Loss(beta=0.08)
    x_tensor = torch.from_numpy(x_train)
    y_tensor = torch.from_numpy(y_train)
    n = len(x_train)

    model.train()
    for _ in range(EPOCHS):
        if time.monotonic() > deadline:
            break
        order = torch.randperm(n)
        for start in range(0, n, BATCH_SIZE):
            if time.monotonic() > deadline:
                break
            idx = order[start:start + BATCH_SIZE]
            xb = x_tensor[idx].to(device)
            yb = y_tensor[idx].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    preds = []
    model.eval()
    pred_tensor = torch.from_numpy(x_pred)
    with torch.no_grad():
        for start in range(0, len(x_pred), BATCH_SIZE):
            xb = pred_tensor[start:start + BATCH_SIZE].to(device)
            preds.append(model(xb).detach().cpu().numpy())
    out = pred[["date", "instrument"]].copy()
    out["mlp_raw"] = np.concatenate(preds).astype("float64") if preds else 0.0
    return out


def _single_fit_mlp(factor_data, start_date, end_date, deadline):
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    factor_data = factor_data.copy()
    factor_data["date"] = pd.to_datetime(factor_data["date"])
    factor_data = factor_data.sort_values(["instrument", "date"])
    if "change_ratio" not in factor_data.columns:
        return pd.DataFrame(columns=["date", "instrument", "mlp_raw"])

    feature_cols = _select_feature_columns(factor_data)
    if len(feature_cols) < 5:
        return pd.DataFrame(columns=["date", "instrument", "mlp_raw"])

    ranked = _make_rank_features(factor_data, feature_cols)
    ranked["future_return"] = factor_data.groupby("instrument")["change_ratio"].shift(-1).values
    ranked["target"] = ranked.groupby("date")["future_return"].rank(pct=True, method="average") - 0.5

    train_mask = (ranked["date"] < start_ts) & ranked["target"].notna()
    pred_mask = (ranked["date"] >= start_ts) & (ranked["date"] <= end_ts)
    train = ranked.loc[train_mask, ["date", "instrument", "target"] + feature_cols].copy()
    pred = ranked.loc[pred_mask, ["date", "instrument"] + feature_cols].copy()
    if pred.empty:
        return pd.DataFrame(columns=["date", "instrument", "mlp_raw"])
    return _fit_predict_mlp(train, pred, feature_cols, deadline)


def _anchor_signals(bar1m, financial, factorlib, instruments, start_date, end_date):
    sql = f"""
    WITH pool AS (
        SELECT date, instrument
        FROM {instruments}
        WHERE date BETWEEN '{start_date}' AND '{end_date}'
    ),
    minute_ranked AS (
        SELECT
            CAST(date AS DATE) AS trade_date,
            instrument,
            close,
            pre_close,
            volume,
            ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), instrument ORDER BY date ASC) AS rn_open,
            ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), instrument ORDER BY date DESC) AS rn_close,
            COUNT(*) OVER (PARTITION BY CAST(date AS DATE), instrument) AS n_bars
        FROM {bar1m}
        WHERE CAST(date AS DATE) BETWEEN '{start_date}' AND '{end_date}'
    ),
    intraday AS (
        SELECT
            trade_date AS date,
            instrument,
            MAX(CASE WHEN rn_open = 1 THEN close END) AS first_close,
            MAX(CASE WHEN rn_open = 1 THEN pre_close END) AS pre_close,
            MAX(CASE WHEN rn_open = 30 THEN close END) AS close_30m,
            MAX(CASE WHEN rn_close = 30 THEN close END) AS pre_tail_close,
            MAX(CASE WHEN rn_close = 1 THEN close END) AS last_close,
            SUM(CASE WHEN rn_close <= 30 THEN volume ELSE 0 END) AS tail_volume,
            SUM(volume) AS total_volume,
            MAX(close) AS high_close,
            MIN(close) AS low_close
        FROM minute_ranked
        GROUP BY trade_date, instrument
    ),
    fin_ttm_ranked AS (
        SELECT
            p.date,
            p.instrument,
            f.operating_revenue,
            f.operating_profit,
            f.net_profit,
            f.net_cffoa,
            ROW_NUMBER() OVER (
                PARTITION BY p.date, p.instrument
                ORDER BY f.date DESC, f.report_date DESC
            ) AS rn
        FROM pool p
        LEFT JOIN {financial} f
          ON p.instrument = f.instrument
         AND f.date <= p.date
         AND f.category = 'ttm'
    ),
    fin_ttm AS (SELECT * FROM fin_ttm_ranked WHERE rn = 1),
    fin_lf_ranked AS (
        SELECT
            p.date,
            p.instrument,
            f.total_assets,
            f.total_liabilities,
            f.interest_bearing_debt,
            f.accounts_receivable,
            f.inventories,
            ROW_NUMBER() OVER (
                PARTITION BY p.date, p.instrument
                ORDER BY f.date DESC, f.report_date DESC
            ) AS rn
        FROM pool p
        LEFT JOIN {financial} f
          ON p.instrument = f.instrument
         AND f.date <= p.date
         AND f.category = 'lf'
    ),
    fin_lf AS (SELECT * FROM fin_lf_ranked WHERE rn = 1),
    daily AS (
        SELECT
            date,
            instrument,
            turn,
            change_ratio,
            reversal_5,
            momentum_5,
            volatility_5,
            netflow_amount_rate_main,
            netflow_amount_main,
            beta_000300SH_22
        FROM {factorlib}
        WHERE date BETWEEN '{start_date}' AND '{end_date}'
    ),
    raw AS (
        SELECT
            p.date,
            p.instrument,
            COALESCE(i.tail_volume, 0) / NULLIF(COALESCE(i.total_volume, 0), 0) AS tail_share,
            COALESCE(d.change_ratio, COALESCE(i.last_close, i.first_close) / NULLIF(i.pre_close, 0) - 1.0) AS day_ret,
            COALESCE(i.first_close, i.pre_close) / NULLIF(i.pre_close, 0) - 1.0 AS open_ret,
            COALESCE(i.last_close, COALESCE(i.pre_tail_close, i.first_close)) / NULLIF(COALESCE(i.pre_tail_close, i.first_close), 0) - 1.0 AS tail_repair,
            COALESCE(i.last_close, COALESCE(i.close_30m, i.first_close)) / NULLIF(COALESCE(i.close_30m, i.first_close), 0) - 1.0 AS post_open_repair,
            COALESCE(i.high_close, i.last_close) / NULLIF(i.low_close, 0) - 1.0 AS intraday_range,
            COALESCE(t.net_cffoa, 0) / NULLIF(ABS(COALESCE(l.total_assets, 0)) + 1.0, 0) +
            0.60 * COALESCE(t.operating_profit, 0) / NULLIF(ABS(COALESCE(l.total_assets, 0)) + 1.0, 0) -
            0.40 * COALESCE(l.total_liabilities, 0) / NULLIF(ABS(COALESCE(l.total_assets, 0)) + 1.0, 0) -
            0.30 * COALESCE(l.interest_bearing_debt, 0) / NULLIF(ABS(COALESCE(l.total_assets, 0)) + 1.0, 0) AS financial_quality,
            COALESCE(t.net_cffoa, 0) / NULLIF(ABS(COALESCE(t.net_profit, 0)) + 1.0, 0) +
            0.40 * COALESCE(t.operating_profit, 0) / NULLIF(ABS(COALESCE(t.operating_revenue, 0)) + 1.0, 0) -
            0.20 * COALESCE(l.accounts_receivable, 0) / NULLIF(ABS(COALESCE(t.operating_revenue, 0)) + 1.0, 0) -
            0.20 * COALESCE(l.inventories, 0) / NULLIF(ABS(COALESCE(t.operating_revenue, 0)) + 1.0, 0) -
            0.25 * COALESCE(l.total_liabilities, 0) / NULLIF(ABS(COALESCE(l.total_assets, 0)) + 1.0, 0) AS cashflow_quality,
            COALESCE(d.turn, 0) AS turn,
            COALESCE(d.reversal_5, 0) AS reversal_5,
            COALESCE(d.momentum_5, 0) AS momentum_5,
            COALESCE(d.volatility_5, 0) AS volatility_5,
            COALESCE(d.netflow_amount_rate_main, 0) AS netflow_amount_rate_main,
            COALESCE(d.beta_000300SH_22, 0) AS beta_000300SH_22
        FROM pool p
        LEFT JOIN intraday i ON p.date = i.date AND p.instrument = i.instrument
        LEFT JOIN daily d ON p.date = d.date AND p.instrument = d.instrument
        LEFT JOIN fin_ttm t ON p.date = t.date AND p.instrument = t.instrument
        LEFT JOIN fin_lf l ON p.date = l.date AND p.instrument = l.instrument
    ),
    anchors AS (
        SELECT
            date,
            instrument,
            tail_share * (-1.0 * day_ret) *
                (1.0 + 0.45 * financial_quality / (1.0 + ABS(financial_quality))) *
                (1.0 + 0.30 * turn) /
                (1.0 + ABS(volatility_5) + ABS(intraday_range)) AS anchor_f921,
            tail_share * (-1.0 * day_ret) * (1.0 + turn) /
                (1.0 + ABS(day_ret)) AS anchor_f906,
            (-0.52 * netflow_amount_rate_main * (1.0 + ABS(momentum_5)) *
                CASE WHEN momentum_5 > 0 THEN 1.35 ELSE 1.00 END +
             0.24 * reversal_5 +
             0.16 * (-1.0 * open_ret) +
             0.10 * tail_repair -
             0.05 * beta_000300SH_22) /
                (1.0 + ABS(volatility_5) + 0.28 * ABS(beta_000300SH_22) + 0.12 * ABS(netflow_amount_rate_main)) AS anchor_f914,
            0.42 * cashflow_quality / (1.0 + ABS(cashflow_quality)) * (-1.0 * netflow_amount_rate_main) +
                0.22 * reversal_5 -
                0.14 * momentum_5 +
                0.14 * tail_repair +
                0.08 * (-1.0 * day_ret) AS anchor_f924
        FROM raw
    )
    SELECT
        date,
        instrument,
        COALESCE(anchor_f921, 0.0) AS anchor_f921,
        COALESCE(anchor_f906, 0.0) AS anchor_f906,
        COALESCE(anchor_f914, 0.0) AS anchor_f914,
        COALESCE(anchor_f924, 0.0) AS anchor_f924
    FROM anchors
    """
    anchors = dai.query(sql, filters={"date": [start_date, end_date]}).df()
    anchors["date"] = pd.to_datetime(anchors["date"])
    return anchors


def main(datasources, start_date, end_date):
    bar1m = datasources["bar1m"]
    financial = datasources["financial"]
    factorlib = "bigalpha_2026_factorlib"
    instruments = "bigalpha_2026_instruments"
    deadline = time.monotonic() + TRAIN_TIME_BUDGET_SECONDS

    start_ts = pd.Timestamp(start_date)
    train_start = (start_ts - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d 00:00:00")
    factor_data = dai.query(
        f"SELECT * FROM {factorlib}",
        filters={"date": [train_start, end_date]},
    ).df()
    pool = dai.query(
        f"SELECT date, instrument FROM {instruments}",
        filters={"date": [start_date, end_date]},
    ).df()
    if factor_data.empty or pool.empty:
        pool["factor"] = 0.0
        return pool[["date", "instrument", "factor"]]

    pool["date"] = pd.to_datetime(pool["date"])
    mlp_pred = _single_fit_mlp(factor_data, start_date, end_date, deadline)
    anchors = _anchor_signals(bar1m, financial, factorlib, instruments, start_date, end_date)
    result = pool.merge(anchors, on=["date", "instrument"], how="left")
    result = result.merge(mlp_pred, on=["date", "instrument"], how="left")

    for col in ["anchor_f921", "anchor_f906", "anchor_f914", "anchor_f924", "mlp_raw"]:
        result[col] = result[col].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        result = _rank_by_date(result, col, col + "_rank")

    stable_anchor = (
        0.50 * result["anchor_f921_rank"].fillna(0.0) +
        0.25 * result["anchor_f906_rank"].fillna(0.0) +
        0.17 * result["anchor_f914_rank"].fillna(0.0) +
        0.08 * result["anchor_f924_rank"].fillna(0.0)
    )
    model_component = np.tanh(1.8 * result["mlp_raw_rank"].fillna(0.0))
    result["factor"] = 0.60 * stable_anchor + 0.40 * model_component
    result["factor"] = result.groupby("date")["factor"].rank(pct=True, method="average") - 0.5
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return result[["date", "instrument", "factor"]]


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时只构造平台会注入的基础数据源映射（逻辑名固定为 "bar1m"/"financial"）
    # 其他允许表在 main 内直接写物理表名，避免提交环境没有替换映射导致失败。
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    # 本地用这段区间模拟「平台注入的测试集区间」（训练区间已在 main 内写死）
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


In [ ]:
if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时只构造平台会注入的基础数据源映射（逻辑名固定为 "bar1m"/"financial"）
    # 其他允许表在 main 内直接写物理表名，避免提交环境没有替换映射导致失败。
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    start_date = '2020-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，全量测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，全量测试区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result_full = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
